# Pre-train Mini BERT on a free Colab GPU

This trains a 4-layer BERT (11 million parameters, the size of Google's BERT-Mini) **from scratch** on WikiText-103, about 100 million words of Wikipedia. It learns by hiding 15% of the words and guessing them.

**To run:**
1. **Runtime → Change runtime type → T4 GPU → Save**
2. **Runtime → Run all**, and allow access to Google Drive when asked.

The first run downloads and tokenizes the text (about 5 minutes), then trains. Every few hundred steps a line shows the loss and the time left. Keep this tab open while it trains.

**If Colab disconnects** or the free GPU time runs out, nothing is lost: progress is saved to `MyDrive/mini-bert/` every 2,000 steps. Come back later and **Run all** again, and it continues from the last save.

When training finishes, the last cells compare the model with Google's BERT-Mini and BERT-Base, and save `results.json` and `training.png` to the same folder.

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU. Runtime → Change runtime type → T4 GPU, then Runtime → Run all."
print(torch.cuda.get_device_name(0), "| PyTorch", torch.__version__)

## 1. Drive and code

In [ ]:
import importlib
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive
from huggingface_hub import hf_hub_download

drive.mount("/content/drive")
HOME = Path("/content/drive/MyDrive/mini-bert")  # the code, the data cache and the training run live here
RUN = HOME / "run-mini"
CODE = Path("/content/mini-bert")

# Unpack the code, add Google's BERT vocabulary (downloaded rather than stored in Drive), install.
shutil.rmtree(CODE, ignore_errors=True)
shutil.unpack_archive(HOME / "mini-bert-code.zip", CODE)
vocab = CODE / "minibert" / "data" / "bert-base-uncased-vocab.txt"
vocab.parent.mkdir(parents=True, exist_ok=True)
shutil.copy(hf_hub_download("bert-base-uncased", "vocab.txt"), vocab)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", str(CODE)], check=True)
importlib.invalidate_caches()

from minibert.tokenizer import Tokenizer  # noqa: E402  (only importable once installed)

tokenizer = Tokenizer.bert_base_uncased(special_in_text=False)
print("Code ready. Vocabulary:", len(tokenizer), "tokens")

## 2. Training text (prepared once, then reused)

In [ ]:
import pyarrow.parquet as pq

from minibert.data import Corpus, prepare_corpus

LOCAL = Path("/content/data")  # training reads the tokens from the fast local disk, not from Drive
CACHE = HOME / "data"
LOCAL.mkdir(exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)
SPLITS = {
    "train": ["train-00000-of-00002.parquet", "train-00001-of-00002.parquet"],
    "valid": ["validation-00000-of-00001.parquet"],
    "test": ["test-00000-of-00001.parquet"],
}


def wikitext_lines(parts):
    for part in parts:
        path = hf_hub_download("Salesforce/wikitext", f"wikitext-103-raw-v1/{part}", repo_type="dataset")
        yield from (line for line in pq.read_table(path).column("text").to_pylist() if line.strip())


for split, parts in SPLITS.items():
    local, cached = LOCAL / f"{split}.bin", CACHE / f"{split}.bin"
    if local.exists():
        continue
    if cached.exists():
        shutil.copy(cached, local)
        shutil.copy(cached.with_suffix(".json"), local.with_suffix(".json"))
        print(f"{split}: copied from Drive")
        continue
    counts = prepare_corpus(wikitext_lines(parts), tokenizer, local, wikitext=True)
    shutil.copy(local, cached)
    shutil.copy(local.with_suffix(".json"), cached.with_suffix(".json"))
    print(f"{split}: {counts['tokens']:,} tokens from {counts['documents']:,} articles")

train, valid, test = (Corpus.open(LOCAL / f"{split}.bin") for split in SPLITS)
print(f"Training text: {len(train):,} tokens")

## 3. Train

60,000 steps of 128 sequences × 128 tokens: about 1 billion tokens, or 9 passes over WikiText-103. The learning rate warms up for 3,000 steps, then falls to zero.

Running this cell again after a disconnect continues from the last checkpoint.

In [ ]:
from minibert.model import MaskedLanguageModel, count_parameters
from minibert.pretrain import TrainSettings, mini_config, pretrain

settings = TrainSettings(
    steps=60_000,
    batch_size=128,
    sequence_length=128,
    learning_rate=5e-4,
    warmup_steps=3_000,
    eval_every=2_000,
    save_every=2_000,
    log_every=500,
)
torch.manual_seed(settings.seed)
model = MaskedLanguageModel(mini_config(len(tokenizer)))
print(f"{count_parameters(model):,} parameters")
result = pretrain(model, tokenizer, train, valid, RUN, settings)
result

## 4. Results

In [ ]:
import json

import matplotlib.pyplot as plt

from minibert.checkpoint import load
from minibert.predict import fill_mask
from minibert.pretrain import evaluate, evaluation_set, read_log

assert result["finished"], "Training isn't finished yet: Runtime → Run all to continue."
log = read_log(RUN)
inputs, labels = evaluation_set(test, tokenizer, 1_000, 128)  # test text: never used during training
scores = {}
for name, source in [
    ("ours (Mini BERT)", RUN / "model"),
    ("Google BERT-Mini", "google/bert_uncased_L-4_H-256_A-4"),
    ("Google BERT-Base", "bert-base-uncased"),
]:
    candidate, _ = load(source)
    scores[name] = evaluate(candidate.cuda(), inputs, labels)
    print(f"{name:<18} guesses {scores[name]['accuracy']:.1%} of hidden words right (loss {scores[name]['loss']:.3f})")

ours, ours_tokenizer = load(RUN / "model")
examples = [
    "Paris is the [MASK] of France.",
    "Winnipeg is the capital of [MASK].",
    "She opened the [MASK] and took out the milk.",
    "The [MASK] won the game in the final minute.",
]
fills = {
    text: [(g.token, round(g.probability, 3)) for g in fill_mask(ours, ours_tokenizer, text)[0]] for text in examples
}
for text, guesses in fills.items():
    print(text, "→", ", ".join(f"{t} {p:.0%}" for t, p in guesses[:3]))

steps = [r["step"] for r in log if "loss" in r]
seconds = sum(
    settings.log_every * settings.batch_size * settings.sequence_length / r["tokens_per_second"]
    for r in log
    if "loss" in r
)
summary = {
    "settings": settings.__dict__,
    "parameters": count_parameters(ours),
    "gpu": torch.cuda.get_device_name(0),
    "training_minutes": round(seconds / 60, 1),
    "median_tokens_per_second": sorted(r["tokens_per_second"] for r in log if "loss" in r)[len(steps) // 2],
    "test": scores,
    "fill": fills,
    "log": log,
}
(HOME / "results.json").write_text(json.dumps(summary, indent=2))

figure, (left, right) = plt.subplots(1, 2, figsize=(11, 4))
left.plot(steps, [r["loss"] for r in log if "loss" in r], label="training")
left.plot(
    [r["step"] for r in log if "eval_loss" in r], [r["eval_loss"] for r in log if "eval_loss" in r], label="held-out"
)
left.set(xlabel="step", ylabel="loss (lower is better)", title="Loss")
left.legend()
right.plot(
    [r["step"] for r in log if "eval_accuracy" in r],
    [100 * r["eval_accuracy"] for r in log if "eval_accuracy" in r],
    label="ours",
)
right.axhline(100 * scores["Google BERT-Mini"]["accuracy"], linestyle="--", color="gray", label="Google BERT-Mini")
right.set(xlabel="step", ylabel="% of hidden words guessed right", title="Held-out accuracy")
right.legend()
figure.tight_layout()
figure.savefig(HOME / "training.png", dpi=150)
print(f"Saved results.json and training.png to {HOME}. Training took about {seconds / 3600:.1f} hours of GPU time.")